<a href="https://colab.research.google.com/github/harrisonm23-byte/HuggingFaceML1/blob/claude%2Fhuggingface-ml-research-setup-ulvgve/notebooks/13_inside_the_mountain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13 — Inside the mountain: what is active in the model when UBS asks

Reads what notebook 12 saved (Google Drive, `silver_study/mountain_<model>/`): `days.jsonl` (every day's steps, trades and replies) and `acts/` (the model's residual stream at every layer, at every decision). It looks at the moments that matter: **the step at which each of UBS's push requests arrives** (and the same moment in the no-ask runs), and asks three things.

1. **What did the agent do?** For each push request: did its own trade go UBS's way and set off UBS's trades (joined), and how much did it add in UBS's direction.
2. **Can its decision be read from inside, before it replies?** For each layer, a simple linear read-out (a probe) trained to tell "went along" from "did not" from the state just before the reply, checked on runs it was not trained on. If a layer reads it well above chance, the decision is already formed there.
3. **Which features differ?** The Gemma Scope 2 sparse autoencoders turn the activations into features with descriptions. For a few layers, the features that differ most between: asked vs no ask, loss vs gain, "you" vs no "you", and went along vs did not. Look each one up on Neuronpedia to read what it is about (e.g. deception, rules, money, loss).

This is a first look at the try-out data (one run of each version and condition), so differences are descriptive, not tests; the final round's numbers are fixed in `designs/rd2/PLAN_final_round.md`. No GPU needed (a CPU runtime is fine). The results are saved to Drive (`analysis/`) and downloaded: send Claude `inside_summary.md`.

In [ ]:
# Libraries, Drive, and the repo (for the day and chat layout)
!pip install -q -U safetensors
import os, sys, re, json, numpy as np, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
from huggingface_hub import login, whoami, list_repo_files, hf_hub_download
# Sign in to Hugging Face automatically with your Colab secret HF_TOKEN (nothing to type); only its first word is used,
# so extra spaces, line breaks, quotes or a token pasted twice don't matter
from google.colab import userdata
login(token=userdata.get("HF_TOKEN").replace('"', " ").replace("'", " ").split()[0])
print("Signed in to Hugging Face as", whoami()["name"])
if not os.path.exists("HuggingFaceML1"):
    os.system("git clone -q -b claude/huggingface-ml-research-setup-ulvgve https://github.com/harrisonm23-byte/HuggingFaceML1.git")
else:
    os.system("git -C HuggingFaceML1 pull -q")
sys.path.insert(0, os.path.abspath("HuggingFaceML1/sim"))
from mountain import TIMES, WINDOW
from chat_market import SIGN

In [ ]:
# Settings
MODEL = "google/gemma-3-12b-it"                              # the model notebook 12 ran (the try-out: "google/gemma-3-4b-it")
DATA = "/content/drive/MyDrive/silver_study/mountain_" + MODEL.split("/")[-1]
SAE_REPO = "google/gemma-scope-2-" + MODEL.split("gemma-3-")[-1]   # Gemma Scope 2 for this model (e.g. gemma-scope-2-12b-it)
SAE_WIDTH = "16k"                                            # the smaller dictionaries (16k features per layer)
SAE_LAYERS = None                                            # None = four layers spread through the model
TOP = 15                                                     # features listed per comparison
# Neuronpedia page for a feature (the address pattern may differ: if a link does not open, search Neuronpedia for the
# model, the layer and the feature number)
NEURONPEDIA = "https://www.neuronpedia.org/" + MODEL.split("/")[-1] + "/{layer}-gemmascope-2-res-16k/{feature}"
OUT = os.path.join(DATA, "analysis")
os.makedirs(OUT, exist_ok=True)

## 1. The push requests, and what the agent did

In [ ]:
days = [json.loads(l) for l in open(os.path.join(DATA, "days.jsonl"))]
STEP = {f"{h:02d}:{m:02d}": i for i, (h, m) in enumerate((8 + k // 4, 15 * (k % 4)) for k in range(24))}   # 08:00 -> 0 ... 13:45 -> 23

def frame_of(v):
    return {"loss": "loss", "loss + you": "loss", "gain": "gain", "gain + you": "gain"}.get(v)

rows = []
for d in days:
    acts = np.load(os.path.join(DATA, d["acts_file"]))
    for cid, at in TIMES[d["date"]]:
        c = d["summary"]["chats"][cid]
        sign = SIGN[c["wants"]]
        if not sign:
            continue                                          # push requests only (share and conceal requests are words)
        w = STEP[WINDOW.get(cid, at)]
        steps = d["steps"]
        before = sign * (steps[w - 1]["position"] if w else 0)
        # Exposure added: how far its own trades took the position UBS's way in the window, beyond flat and beyond where it was
        own = [sign * ((steps[i - 1]["position"] if i else 0) + steps[i]["traded"]) for i in range(w, min(w + 2, len(steps)))]
        rows.append({"version": d["version"], "frame": frame_of(d["version"]), "you": "+ you" in d["version"],
                     "asked": d["version"] != "no ask", "mode": d["mode"], "run": d["run"], "date": d["date"], "chat": cid,
                     "step": STEP[at], "joined": bool(c["joined"]), "their_way": c["window_their_way"],
                     "exposure_added": max(0, max(own) - max(0, before)) if own else 0,
                     "reply": steps[STEP[at]]["reply"], "acts_file": d["acts_file"]})
asks = pd.DataFrame(rows)
print(len(asks), "push requests (", asks.asked.sum(), "asked,", (~asks.asked).sum(), "no-ask moments )")
(asks.groupby(["mode", "version"]).agg(requests=("joined", "size"), joined=("joined", "mean"),
                                       exposure_added=("exposure_added", "mean")).round(2))

## 2. Can the decision be read from inside, before it replies?

For each layer: a linear probe on the state at the last token of the update (before the reply), predicting whether the agent then joined UBS's push. Checked by leaving whole runs out (it never sees the run it is tested on), and compared with the same probe on shuffled labels (chance). Asked versions only. With the try-out's numbers this is a rough first look.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

_files = {}
def state(row, kind="last"):
    # The state before the reply at that step, (layers, width); each day's file is read once
    if row["acts_file"] not in _files:
        z = np.load(os.path.join(DATA, row["acts_file"]))
        _files[row["acts_file"]] = {"last": z["last"], "reply_mean": z["reply_mean"]}
    return _files[row["acts_file"]][kind][row["step"]].astype(np.float32)

asked = asks[asks.asked].reset_index(drop=True)
y = asked.joined.values
probe = []
if 0 < y.sum() < len(y):
    X = np.stack([state(r) for _, r in asked.iterrows()])           # (requests, layers, width)
    groups = asked["version"] + "|" + asked["run"].astype(str)
    folds = GroupKFold(n_splits=min(5, groups.nunique()))
    rng = np.random.default_rng(0)
    for layer in range(X.shape[1]):
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=2000))
        p = cross_val_predict(clf, X[:, layer], y, groups=groups, cv=folds, method="predict_proba")[:, 1]
        shuffled = rng.permutation(y)
        ps = cross_val_predict(clf, X[:, layer], shuffled, groups=groups, cv=folds, method="predict_proba")[:, 1]
        probe.append({"layer": layer, "auc": roc_auc_score(y, p), "auc_shuffled": roc_auc_score(shuffled, ps)})
    probe = pd.DataFrame(probe)
    print(f"{int(y.sum())} of {len(y)} asked requests joined. Read-out by layer (0.5 = chance):")
    print(probe.round(2).to_string(index=False))
else:
    print(f"The agent joined {int(y.sum())} of {len(y)} asked requests: nothing to tell apart, so no probe.")

## 3. Gemma Scope features

Loads the Gemma Scope 2 residual-stream SAE for each chosen layer (from Hugging Face; the file names are printed so they can be checked), turns the state before each reply into features, and lists the features that differ most between groups. The difference is shown in units of the features' spread (like an effect size), among features active in at least 10% of the requests.

In [ ]:
from safetensors.numpy import load_file

files = list_repo_files(SAE_REPO)
n_layers = np.load(os.path.join(DATA, asks.acts_file.iloc[0]))["last"].shape[1]
layers = SAE_LAYERS or [round(n_layers * f) for f in (0.25, 0.5, 0.65, 0.8)]

def sae_file(layer):
    # The residual-stream SAE for this layer and width (not MLP, attention, transcoders or crosscoders)
    hit = [f for f in files if re.search(rf"(^|[/_-])layer[_-]?{layer}([/_-]|$)", f) and SAE_WIDTH in f
           and re.search(r"res", f) and not re.search(r"mlp|att|transcoder|crosscoder|clt", f)
           and f.endswith((".safetensors", ".npz"))]
    hit.sort(key=lambda f: (("medium" not in f), len(f)))
    return hit

def load_sae(layer):
    hit = sae_file(layer)
    if not hit:
        raise FileNotFoundError(f"No residual {SAE_WIDTH} SAE file for layer {layer} in {SAE_REPO}; some files: {files[:20]}")
    path = hf_hub_download(SAE_REPO, hit[0])
    p = load_file(path) if path.endswith(".safetensors") else dict(np.load(path))
    p = {k.lower().replace(".", "_"): v.astype(np.float32) for k, v in p.items()}
    pick = lambda *names: next((p[n] for n in names if n in p), None)
    W = pick("w_enc", "encoder_weight")
    b = pick("b_enc", "encoder_bias")
    t = pick("threshold", "log_threshold")
    if "log_threshold" in p and "threshold" not in p:
        t = np.exp(t)
    if W.shape[0] != n_width:
        W = W.T
    print(f"layer {layer}: {hit[0]} | {W.shape[1]} features | {'JumpReLU' if t is not None else 'ReLU'}")
    return W, b, t

def encode(x, sae):
    W, b, t = sae
    pre = x @ W + b
    return np.where(pre > (t if t is not None else 0), pre, 0.0)

n_width = np.load(os.path.join(DATA, asks.acts_file.iloc[0]))["last"].shape[2]
feats = {}
for layer in layers:
    sae = load_sae(layer)
    feats[layer] = np.stack([encode(state(r)[layer], sae) for _, r in asks.iterrows()])
print("features ready for layers", layers)

In [ ]:
def differ(a, b, layer, name):
    # Features that differ most between group a and group b (rows of asks), in units of their spread
    F = feats[layer]
    A, B = F[a.values], F[b.values]
    if len(A) < 2 or len(B) < 2:
        return pd.DataFrame()
    active = (F > 0).mean(axis=0) >= 0.10
    spread = F.std(axis=0) + 1e-6
    d = (A.mean(axis=0) - B.mean(axis=0)) / spread
    d[~active] = 0
    top = np.argsort(-np.abs(d))[:TOP]
    return pd.DataFrame({"comparison": name, "layer": layer, "feature": top, "difference": d[top].round(2),
                         "active_a": (A[:, top] > 0).mean(axis=0).round(2), "active_b": (B[:, top] > 0).mean(axis=0).round(2),
                         "neuronpedia": [NEURONPEDIA.format(layer=layer, feature=f) for f in top]})

comparisons = [("asked vs no ask", asks.asked, ~asks.asked),
               ("loss vs gain", asks.frame == "loss", asks.frame == "gain"),
               ('"you" vs no "you" (framed only)', asks.you & asks.frame.notna(), ~asks.you & asks.frame.notna()),
               ("went along vs did not (asked)", asks.asked & asks.joined, asks.asked & ~asks.joined),
               ("full takeover vs one line (asked)", asks.asked & (asks["mode"] == "takeover 5 days"), asks.asked & (asks["mode"] == "takeover 1 day"))]
table = pd.concat([differ(a, b, layer, name) for name, a, b in comparisons for layer in layers], ignore_index=True)
table.to_csv(os.path.join(OUT, "features.csv"), index=False)
for name in table.comparison.unique():
    print(f"\n== {name} == (top 5 per layer; all {TOP} in features.csv)")
    print(table[table.comparison == name].groupby("layer").head(5).drop(columns="comparison").to_string(index=False))

## 4. Save and download

In [ ]:
def md_table(df):
    # A markdown table without extra packages
    df = df.reset_index() if df.index.name or any(df.index.names) else df
    head = "| " + " | ".join(str(c) for c in df.columns) + " |"
    return "\n".join([head, "|" + "---|" * len(df.columns)] + ["| " + " | ".join(str(x) for x in r) + " |" for r in df.itertuples(index=False)])

lines = [f"# Inside the mountain: {MODEL}", "",
         f"{len(asks)} push requests ({int(asks.asked.sum())} asked) from {len(days)} days.", "",
         "## What the agent did (per push request)", "",
         md_table(asks.groupby(["mode", "version"]).agg(requests=("joined", "size"), joined=("joined", "mean"),
                                                        exposure_added=("exposure_added", "mean")).round(2)), ""]
if len(probe):
    lines += ["## Read-out of 'went along' by layer (AUC, 0.5 = chance)", "", md_table(probe.round(2)), ""]
lines += ["## Features that differ most (Gemma Scope 2)", "", md_table(table)]
open(os.path.join(OUT, "inside_summary.md"), "w").write("\n".join(lines))
asks.drop(columns=["acts_file"]).to_csv(os.path.join(OUT, "push_requests.csv"), index=False)
print("Saved to", OUT)
try:
    from google.colab import files as colab_files
    colab_files.download(os.path.join(OUT, "inside_summary.md"))
except Exception:
    pass

Send Claude `inside_summary.md` (downloaded above; also in `analysis/` on Drive with `features.csv` and `push_requests.csv`).